In [ ]:
# Install the package with my custom environment
import sys
!{sys.executable} -m pip install -e /home/a/A.Rivera/Code_Projects/1_Thesis_Code/TASEP_2D/SmartTASEP/gym-environments

In [ ]:
import gymnasium as gym
import gym_environments 
from gymnasium import spaces
from numba import njit
import numpy as np
import torch
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque

In [ ]:
# our main
if __name__ == '__main__':

    # Simulation parameters
    Lx = 40 # Number of columns, length
    Ly = 20 # Number of rows, height
    N = Lx * Ly // 2
    size = Lx * Ly
    
    runsNumber = 1
    totalMCS = 80
    init = "chess"
    mu = 0.5
    sigma = 0.01
    
    env = gym.make('gym_environments/LatticeTASEP-v0', render_mode="ansi", mode = init, Lx = Lx, Ly = Ly, n = N, max_steps = N, mu = mu, sigma = sigma)            
    System, info = env.reset()
    
    action = 0
    System, reward, terminated, truncated, info = env.step(action)
    
    print(info["Along_Steps"])






In [ ]:
def current_plot(sigma):
    plt.cla()
    plt.title(f"TASEP. Gaussian jumping rate over {runsNumber} runs")
    plt.xlabel('Time')
    plt.ylabel(f'Current. {N} moves averaged')
    plt.grid(True)
    
    #The steady current of particle J, through a bond i, i+1 is given by the rate r multiplied by the probability that there is a particle at site i, and site i+1 is vacant
    r = 0.5 #jumping rate
    p = 0.5 #probability forward
    prob_occ=1 # we make a transition only when the site chosen is occupied
    prob_vac=1/2 #density of particles in the system
    f = np.array([r*p*prob_occ*prob_vac]*totalMCS)
    
    sigma = fixed_sigma
    x_axis=np.array(range(totalMCS))
    CurrentAlongTot, CurrentTransvTot = Simulate(runsNumber, totalMCS, Lx, Ly, init, mu, sigma)    
    
    plt.plot(x_axis[5:], CurrentAlongTot[5:], '.', label='Parallel Current')
    plt.plot(x_axis[5:], CurrentTransvTot[5:], '.', label='Perpendicular Current')
    plt.plot(x_axis, f, label='Parallel Current for p=1/2', color = 'red')
    plt.legend(loc = 'center right')

In [ ]:
#@njit
def Simulate(runsNumber, totalMCS, Lx, Ly, init, mu, sigma):   
    env = gym.make('gym_environments/LatticeTASEP-v0', render_mode="ansi", mode = init, Lx = Lx, Ly = Ly, n = N, max_steps = N, mu = mu, sigma = sigma)                
 # Check utilities
    print_stuff = 0 #0 do not print; 1 print basics; 2 print details; for unit tests
    if print_stuff == 2 and totalMCS > 10:
        raise ValueError("totalMCS too big to print everything")
    SystemCheckboard = np.zeros((Ly, Lx), dtype=np.float32)  # Ly vectors with Lx (zero) components 
    SystemCheckboard[::2, ::2] = 1  # Set even rows and even columns to 1
    SystemCheckboard[1::2, 1::2] = 1  # Set odd rows and odd columns to 1
    
 # Memory allocation  
    CurrentAlongTot = np.zeros(totalMCS, dtype=np.float32)  
    CurrentTransvTot = np.zeros(totalMCS, dtype=np.float32) 

 #Beginning of the simulation
    for iwalk in range(runsNumber):
       # Initialize system
        System, info = env.reset()
        
       # Mapping System at t=0
        SystemSnapshot = System.copy()
        if init == "chess" and print_stuff > 0: 
            if (SystemSnapshot != SystemCheckboard).all(): raise ValueError("System at t = 0, not in checkboard mode")        
        if print_stuff == 2: print(SystemSnapshot)


       # Memory allocation                   
        CurrentAlong = np.zeros(totalMCS, dtype=np.float32) 
        CurrentTransv = np.zeros(totalMCS, dtype=np.float32)
        #current = []

       # Beginning of a single run
        for istep in range(totalMCS):
            Along_count = 0
            Transv_count = 0

            for moveAttempt in range(N): # To make a move over all particles
                while True:
                    action = random.randint(0, Lx * Ly - 1)  # Picks the random lattice site in the array
                    """ 
                    The lattice is labelled from the left top to the right 
                     as: dice  = 0, 1, 2    or    [X,Y]= (0,0), (0,1), (0,2) 
                                 3, 4, 5                (1,0), (1,1), (1,2)
                    """                      
                    X = action // Lx
                    Y = action - X * Lx

                    if System[X][Y] == 1: # The lattice has to be occupied
                        System, reward, terminated, truncated, info = env.step(action)
                        Along_count += info["Along_Steps"]
                        Transv_count += info["Transv_Steps"]
                        break


            # Computes currents
            #CurrentAlong[istep] = current.s
            CurrentAlong[istep] = Along_count / current_averaging_time  # Sum of the current along Lx
            CurrentTransv[istep] = Transv_count / current_averaging_time # Sum of the current along Ly

        for dt in range(totalMCS):
            CurrentAlongTot[dt] += CurrentAlong[dt]
            CurrentTransvTot[dt] += CurrentTransv[dt]

        if print_stuff == 1 or print_stuff == 2: print("Runs done:", iwalk+1)


    # Simulation results output
    for dt in range(totalMCS):
        CurrentAlongTot[dt] /= runsNumber
        CurrentTransvTot[dt] /= runsNumber

    print('One Simulation finished!')
    
    return CurrentAlongTot, CurrentTransvTot

In [ ]:
# our main
if __name__ == '__main__':

    # Simulation parameters
    Lx = 40 # Number of columns, length
    Ly = 20 # Number of rows, height
    N = Lx * Ly // 2
    size = Lx * Ly
    
    runsNumber = 1
    totalMCS = 80
    init = "chess"
    mu = 0.5
    fixed_sigma = 0.01
    current_averaging_time = N
    
    current_plot(fixed_sigma)

In [ ]:
CurrentAlongTot, CurrentTransvTot

In [ ]:
x_axis=np.array(range(totalMCS))

#The steady current of particle J, through a bond i, i+1 is given by the rate r multiplied by the probability that there is a particle at site i, and site i+1 is vacant
r=1
p=1/2
prob_vac=1/2
f2=np.array([r*p*prob_vac]*totalMCS)

plt.plot(x_axis[5:], CurrentAlongTot[5:], '.', label='Parallel Current')
plt.plot(x_axis[5:], CurrentTransvTot[5:], '.', label='Perpendicular Current')

plt.plot(x_axis,f2, label='Parallel Current for p=1/2', color = 'red')

plt.xlabel('MonteCarlo iterations')
plt.ylabel('Current intensity')
plt.legend(loc = 'center right')


plt.grid(True)

#plt.savefig('Random_sigma=0,01 -  Currents.pdf')

plt.show()

In [ ]:
# Current comparison
def currents_sigmas():
   # Plot
    plt.cla()
    plt.title(f"TASEP. Gaussian jumping rate over {runsNumber} runs")
    plt.xlabel('Time')
    plt.ylabel('Current')
    plt.grid(True)

    plt.rcParams['text.usetex'] = True
    
    for sigma in [0.0001, 0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 5.0]:
        DensityParticlesTot, CorrTot, CurrentAlongTot, CurrentTransvTot, HorizontalOccupProb = Simulate(runsNumber, totalMCS, Lx, Ly, mu, sigma)    
        times = np.array(range(totalMCS))     
        plt.plot(times, CurrentAlongTot, label=f"$\\sigma = {sigma}$")
        
    plt.legend()
    #plt.savefig('Pictures/Current_Different_Sigmas/Currents.pdf')
    
    
@njit
def average_run_current(sigmas):
    currents = np.zeros(sigmas.shape[0], dtype = np.float32)    
    i = 0
    for sigma in sigmas:
        DensityParticlesTot, CorrTot, CurrentAlongTot, CurrentTransvTot, HorizontalOccupProb = Simulate(runsNumber, totalMCS, Lx, Ly, mu, sigma)    
        currents[i] = np.mean(CurrentAlongTot)
        i += 1
        print(i)
    return currents

def average_run_current_over_sigmas():
   # Plot
    plt.cla()
    plt.xlabel('Standard Deviation, $\\sigma$ (log scale)')
    plt.xscale('log')
    plt.ylabel('Current')
    plt.title(f"TASEP. Gaussian jumping rate over {runsNumber} runs")
    plt.grid(True)

    plt.rcParams['text.usetex'] = True        
    
    sigmas = np.logspace(-4, 1, 150, dtype=np.float32)        
    currents = average_run_current(sigmas)
    plt.plot(sigmas, currents)
    #plt.savefig('Pictures/Current_Different_Sigmas/AverageCurrentVsSigma.pdf')      

In [ ]:
HorizontalOccupProb

In [ ]:
initial_obs, info = env.reset()
initial_obs

In [ ]:
action = torch.tensor([[env.action_space.sample()]], dtype=torch.long)    
action.item()

In [ ]:
observation, reward, terminated, truncated, _ = env.step(2)
print(observation, reward, terminated, truncated, _)
print(initial_obs[1])

In [ ]:
env = gym.make('gym_environments/Lattice-v0', render_mode="ansi", mode = "chess", Lx = 2, Ly = 4, n = 4, max_steps = 4, mu = 0.5, sigma = 0.3)

In [ ]:
print("observation_space:", env.observation_space)
print("reward_range:", env.reward_range)
print("metadata:", env.metadata)
print("action_space:", env.action_space)

In [ ]:
initial_obs, info = env.reset()
initial_obs

In [ ]:
action = torch.tensor([[env.action_space.sample()]], dtype=torch.long)    
action.item()

In [ ]:
observation, reward, terminated, truncated, _ = env.step(3)
print(observation, reward, terminated, truncated, _)
print(initial_obs)

In [ ]:
env.unwrapped.particle_current()

In [ ]:
random.seed(124)

In [ ]:
for _ in range(20):
    print(random.randint(0, 3))

In [ ]:
def count_particles(board):     # Computes the total number of particles
## Aún queda comorbar si funciona con el mapa random con número de agentes aleatorios
    NumberParticles = sum([sum(row) for row in board])
    return NumberParticles  

def correlation_function(initial_board, board, Lx, Ly):
    DensityParticles = count_particles(board)/(Lx*Ly)
    Sum = 0
    for i in range(Lx):
        for j in range(Ly):
            Sum += board[i][j] * initial_board[i][j] # Correlation with the snapshot of the system at t = 0
    Corr = Sum / (Lx*Ly) - DensityParticles**2  # In chess fashion the density of particle^2 is 0.25        
    return Corr, Sum

In [ ]:
env = gym.make('gym_environments/Lattice-v0', render_mode="rgb_array", desc = "chess", Lx=6, Ly=6, n=1,  max_steps = 3)

In [ ]:
from itertools import count


# random policy implementation (no learning)
def select_action_random():
    
# 2, 0, 1, 0, 3, 2, 2, 0, 2, 1, 3, 2, 3, 0, 3, 3, 0, 1, 1, 0
    return torch.tensor([[random.randint(0, 3)]], dtype=torch.long)

def count_particles(board):     # Computes the total number of particles
## Aún queda comorbar si funciona con el mapa random con número de agentes aleatorios
    NumberParticles = sum([sum(row) for row in board])
    return NumberParticles  


def correlation_function(initial_board, board, Lx, Ly):
    DensityParticles = count_particles(board)/(Lx*Ly)
    Sum = 0
    for i in range(Lx):
        for j in range(Ly):
            Sum += board[i][j] * initial_board[i][j] # Correlation with the snapshot of the system at t = 0
    Corr = Sum / (Lx*Ly) - DensityParticles**2  # In chess fashion the density of particle^2 is 0.25        
    return Corr

In [ ]:
episode_durations = []
random.seed(123)
Lx = 6
Ly = 6

num_episodes = 5
state, info = env.reset()
initial_state = state.copy()

# Memory allocation
Rewards = [0] * num_episodes  
Corr = [0.0] * num_episodes
DensityParticles = [0] * num_episodes
CurrentAlong = [0] * num_episodes 
CurrentTransv = [0] * num_episodes          

for i_episode in range(num_episodes): # num_episodes of 50 movements or actions
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(np.reshape(state, (1, Lx*Ly)), dtype=torch.float32)

    for t in count(): # It does what it says, it counts

        #Agent-Enviroment Loop = Agent interacts with the enviroment            
        #action = select_action(state) # Greedy epsilon policy
        action = select_action_random() # Random policy
        observation, reward, terminated, truncated, _ = env.step(action.item())
        print(action.item())

        reward = torch.tensor([reward])
        done = terminated or truncated # It always does 10 movements per episode

        if done:
            next_state = None
        else:
            next_state = torch.tensor(np.reshape(observation, (1, Lx*Ly)), dtype=torch.float32)

        # Move to the next state
        state = next_state

        if done:
            Rewards[i_episode] = reward
            DensityParticles[i_episode] = count_particles(observation)/ (Lx*Ly)
            Corr[i_episode] = correlation_function(initial_state, observation, Lx, Ly)
            CurrentAlong[i_episode] = env.unwrapped.particle_current()[0]/3
            CurrentTransv[i_episode] = env.unwrapped.particle_current()[1]/3 

            episode_durations.append(t + 1)
            break

In [ ]:
print(observation)

In [ ]:
CurrentAlong

In [ ]:
CurrentTransv

In [ ]:
CurrentAlong = CurrentAlong_temp.sum(axis=0) # Sum of the current in each lattice in Ly
CurrentTransv = CurrentTransv_temp.sum(axis=0) # Sum of the current in each lattice in Lx

In [ ]:
CurrentAlong

In [ ]:
state.size()

In [ ]:
state = [-0.02943565, -0.04105512, -0.04193077, -0.04223724]
state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
state

In [ ]:
state.size()

In [ ]:
state = torch.tensor(state, dtype=torch.float32).view(1,1) #As the state is an integer instead of a vector, we change unsqueeze (0) by view (1,

In [ ]:
num_episodes = 5

In [ ]:
Rewards = [0.0] * num_episodes  
Corr = [0.0] * num_episodes  
NumberParticles = [0.0] * num_episodes
CurrentAlong = [0.0] * num_episodes 
CurrentTransv = [0.0] * num_episodes   

In [ ]:
Rewards

In [ ]:
for dt in range(num_episodes):
    Rewards[dt] = 0
    Corr[dt] = 0
    NumberParticles[dt] = 0        
    CurrentAlong[dt] = 0
    CurrentTransv[dt] = 0   

In [ ]:
Rewards

In [ ]:
import matplotlib.pyplot as plt
env = gym.make('gym_environments/GridWorld-v3', render_mode="rgb_array")
observation_init = env.reset()
# Start a loop to interact with the environment and capture frames
frames = []  # To store the captured frames

# Capture the frame as an RGB array
frame = env.render()

# Append the frame to the list of frames
frames.append(frame)


In [ ]:
for _ in range(3):
    # Replace 'your_action' with the action you want to take in the environment
    action = 0

    # Perform the action in the environment
    observation, reward, done, info = env.step(action)[0], env.step(action)[1], env.step(action)[2], env.step(action)[3]

    # Capture the frame as an RGB array
    frame = env.render()

    # Append the frame to the list of frames
    frames.append(frame)

    # Check if the episode is done and break the loop if necessary
    if done:
        break

In [ ]:
observation_init

In [ ]:
observation

In [ ]:
# Close the environment when you're done
env.close()

# Visualize the captured frames
for frame in frames:
    plt.imshow(frame)
    plt.axis('off')
    plt.show()

In [ ]:
# Close the rendering window when you're done
env.close()

In [ ]:
# Auxiliary information
def _get_info():
    return {"Along_Steps":[],"Transv_Steps":[]}  

In [ ]:
info = _get_info()

In [ ]:
info["Along_Steps"].append(Along_count)
info["Transv_Steps"].append(Transv_count)